# 02 · Calibración del factor VIIRS → MODIS

**Responsables:** Datos (4) y Backend 2 · ML (2).

Este cuaderno toma tres decisiones del método antes de que Datos escriba `harmonize.py`:

1. **Ventana de calibración.** Se comprueba que el cociente Aqua / VIIRS es estable en 2012–2021 y cambia después, cuando Terra y Aqua derivaron de órbita.
2. **Grilla para contar celdas-día:** 0,1°, 0,25° o 0,5°.
3. **Tipo de factor:** uno solo, uno por estación o uno por mes.

`k = celdas-día Aqua / celdas-día VIIRS S-NPP` lleva VIIRS a la escala MODIS. En Argentina se espera k ≈ 0,2.

**Entrada:** cualquier CSV de NASA FIRMS dentro de `ml/data/raw/` (descarga del archivo histórico o API por área con fuentes `_SP`). Se identifican por las columnas `instrument` y `satellite`, no por el nombre del archivo.

**Regla:** ninguna cifra de este cuaderno se escribe a mano. La sección Decisión queda en «pendiente» hasta correrlo con datos reales.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ML = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ML / "data" / "raw"
FIGURES = ML.parent / "docs" / "capturas"
FIGURES.mkdir(parents=True, exist_ok=True)

# Mismos valores que ml/kindra/config.py (Datos); si ya existe, manda ese archivo.
try:
    from kindra.config import CALIB_YEARS, MODIS_MIN_CONF
except ImportError:
    CALIB_YEARS = range(2012, 2022)   # antes de la deriva orbital de Terra y Aqua
    MODIS_MIN_CONF = 30               # MODIS: confianza 0–100
HOLDOUT_YEARS = range(2022, 2026)
VIIRS_START = "2012-01-20"            # inicio de VIIRS S-NPP en FIRMS
RESOLUTIONS = [0.1, 0.25, 0.5]        # grados
AR_BBOX = (-74, -56, -53, -21)        # oeste, sur, este, norte

files = sorted(RAW.rglob("*.csv"))
print(f"{len(files)} archivos CSV en {RAW}")
assert files, "No hay CSV de FIRMS en ml/data/raw/. Pedírselos a Datos (4)."

## 1. Carga

DuckDB lee todos los CSV juntos aunque sus columnas cambien entre MODIS (`brightness`) y VIIRS (`bright_ti4`). Solo se leen los años con los dos sensores: desde el inicio de VIIRS S-NPP hasta 2025.

In [ ]:
con = duckdb.connect()
pattern = str(RAW / "**" / "*.csv").replace("\\", "/")
source = f"read_csv('{pattern}', union_by_name = true, all_varchar = true)"

columns = set(con.sql(f"describe select * from {source}").df()["column_name"])
# La API NRT y algunos CSV no traen `type`; en ese caso no se puede filtrar por tipo.
type_expr = "type" if "type" in columns else "null"

fires = con.sql(f'''
    select cast(latitude as double)  as lat,
           cast(longitude as double) as lon,
           cast(acq_date as date)    as day,
           trim(satellite)           as satellite,
           upper(trim(instrument))   as instrument,
           lower(trim(confidence))   as confidence,
           {type_expr}               as type
    from {source}
    where cast(acq_date as date) between '{VIIRS_START}' and '2025-12-31'
''').df()
fires["day"] = pd.to_datetime(fires["day"])
print(f"{len(fires):,} focos leídos".replace(",", "."))
fires.groupby(["instrument", "satellite"]).size()

## 2. Limpieza

- Solo `type = 0` (vegetación) cuando la columna existe.
- MODIS con confianza ≥ 30; VIIRS con confianza nominal o alta.
- Solo los dos satélites de la tarde que se comparan: **Aqua** y **VIIRS S-NPP**.

FIRMS escribe el satélite de varias formas según el archivo. Si la celda siguiente muestra códigos sin reconocer, hay que agregarlos a `SATELLITES`.

In [ ]:
SATELLITES = {"Aqua": "Aqua", "A": "Aqua", "Terra": "Terra", "T": "Terra",
              "N": "SNPP", "NPP": "SNPP", "Suomi NPP": "SNPP",
              "N20": "NOAA20", "1": "NOAA20", "N21": "NOAA21"}
fires["sat"] = fires["satellite"].map(SATELLITES)
print("Satélites sin reconocer:", sorted(fires.loc[fires.sat.isna(), "satellite"].unique()))

is_modis = fires.instrument.str.startswith("MODIS")
modis_ok = pd.to_numeric(fires.confidence, errors="coerce") >= MODIS_MIN_CONF
# VIIRS usa "n"/"h"/"l" o "nominal"/"high"/"low" según el archivo.
viirs_ok = fires.confidence.str[0].isin(["n", "h"])
type_ok = fires.type.isna() | (fires.type.astype(str) == "0")
west, south, east, north = AR_BBOX
inside = fires.lon.between(west, east) & fires.lat.between(south, north)

clean = fires[type_ok & inside & np.where(is_modis, modis_ok, viirs_ok) & fires.sat.isin(["Aqua", "SNPP"])]
clean = clean[["lat", "lon", "day", "sat"]].copy()
summary = clean.groupby([clean.day.dt.year.rename("año"), "sat"]).size().unstack("sat")
summary

## 3. Celdas-día por mes

Una celda con fuego en un día cuenta una vez por satélite, sin importar cuántos píxeles caigan en ella. Así se quita el efecto del tamaño de píxel (1 km en MODIS, 375 m en VIIRS). Se usan índices enteros de celda para evitar errores de redondeo.

In [ ]:
def monthly_cell_days(points: pd.DataFrame, res: float) -> pd.DataFrame:
    # Devuelve una fila por año y mes con las celdas-día de Aqua (`a`) y de VIIRS (`v`).
    cells = points.assign(i=np.floor(points.lat / res).astype(int),
                          j=np.floor(points.lon / res).astype(int))
    cells = cells.drop_duplicates(["sat", "i", "j", "day"])
    counts = (cells.groupby([cells.day.dt.year.rename("year"), cells.day.dt.month.rename("month"), "sat"])
                   .size().unstack("sat", fill_value=0)
                   .rename(columns={"Aqua": "a", "SNPP": "v"}))
    return counts.reset_index()[["year", "month", "a", "v"]]


counts = {res: monthly_cell_days(clean, res) for res in RESOLUTIONS}
counts[0.1].tail(12)

## 4. ¿Por qué calibrar solo en 2012–2021?

Si el cociente anual Aqua / VIIRS se mantiene parejo en 2012–2021 y se mueve después, la ventana queda justificada: el cambio no es del fuego sino de la hora de paso de Aqua.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for res, c in counts.items():
    yearly = c.groupby("year")[["a", "v"]].sum()
    ax.plot(yearly.index, yearly.a / yearly.v, marker="o", label=f"{res}°")
ax.axvspan(min(CALIB_YEARS) - 0.5, max(CALIB_YEARS) + 0.5, color="#1B2835", alpha=0.06, label="Calibración")
ax.set(xlabel="Año", ylabel="k = Aqua / VIIRS", title="Cociente anual de celdas-día · Argentina")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES / "calibracion_k_anual.png", dpi=200)

## 5. Factor por mes con intervalo

Igual que `harmonize.py`: `k` por mes del año en 2012–2021 e intervalo del 90 % con bootstrap sobre los años. Un intervalo angosto indica que el factor es estable con esa grilla.

In [ ]:
def monthly_factors(c: pd.DataFrame, years=CALIB_YEARS, n_boot: int = 300, seed: int = 42) -> pd.DataFrame:
    # Devuelve k, k_low y k_high por mes; el remuestreo toma años completos.
    c = c[c.year.isin(years)]
    rng = np.random.default_rng(seed)
    rows = []
    for month, g in c.groupby("month"):
        a, v = g.a.to_numpy(), g.v.to_numpy()
        idx = rng.integers(0, len(g), size=(n_boot, len(g)))
        boot = a[idx].sum(axis=1) / np.maximum(v[idx].sum(axis=1), 1)
        rows.append({"month": month, "k": a.sum() / max(v.sum(), 1),
                     "k_low": np.percentile(boot, 5), "k_high": np.percentile(boot, 95),
                     "n_years": len(g), "aqua": int(a.sum()), "viirs": int(v.sum())})
    return pd.DataFrame(rows)


factors = {res: monthly_factors(c) for res, c in counts.items()}

stability = pd.DataFrame({
    res: {"k medio": f.k.mean(),
          "ancho medio del intervalo (% de k)": ((f.k_high - f.k_low) / f.k).mean() * 100,
          "meses con < 50 celdas-día Aqua": int((f.aqua < 50).sum())}
    for res, f in factors.items()}).T
stability.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for res, f in factors.items():
    ax.errorbar(f.month + (res - 0.25) * 0.4, f.k, yerr=[f.k - f.k_low, f.k_high - f.k],
                fmt="o", capsize=2, label=f"{res}°")
ax.set(xticks=range(1, 13), xticklabels=list("EFMAMJJASOND"), xlabel="Mes", ylabel="k",
       title="Factor mensual VIIRS → MODIS, 2012–2021 (intervalo del 90 %)")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES / "calibracion_k_mensual.png", dpi=200)

## 6. ¿Qué tipo de factor generaliza mejor?

Se compara con **validación dejando un año afuera** dentro de 2012–2021: se calibra con nueve años y se predice Aqua en el décimo (`Aqua ≈ VIIRS × k`). Los años 2022–2025 quedan reservados para la validación final de `validation.py`.

- **único:** un solo `k` para todo el año.
- **estacional:** un `k` por estación (verano DEF, otoño MAM, invierno JJA, primavera SON).
- **mensual:** un `k` por mes, como propone el tutorial.

Métricas: error absoluto medio en celdas-día por mes, error del total anual (%) y correlación `r` entre lo predicho y Aqua real.

In [ ]:
SEASON = {12: "DEF", 1: "DEF", 2: "DEF", 3: "MAM", 4: "MAM", 5: "MAM",
          6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}
KINDS = ["único", "estacional", "mensual"]


def fit_k(train: pd.DataFrame, kind: str) -> pd.Series:
    # Devuelve k por mes del año (índice 1–12) según el tipo de factor.
    group = {"único": np.zeros(len(train)), "estacional": train.month.map(SEASON), "mensual": train.month}[kind]
    sums = train.groupby(np.asarray(group))[["a", "v"]].sum()
    k = sums.a / sums.v.clip(lower=1)
    months = pd.Series(range(1, 13), index=range(1, 13))
    keys = {"único": months * 0, "estacional": months.map(SEASON), "mensual": months}[kind]
    return keys.map(k)


def evaluate(c: pd.DataFrame, kind: str, train_years, test_years, leave_one_out: bool) -> dict:
    preds = []
    for year in (test_years if leave_one_out else [None]):
        if leave_one_out:
            train, test = c[c.year.isin(train_years) & (c.year != year)], c[c.year == year]
        else:
            train, test = c[c.year.isin(train_years)], c[c.year.isin(test_years)]
        preds.append(test.assign(pred=test.v * test.month.map(fit_k(train, kind))))
    p = pd.concat(preds)
    yearly = p.groupby("year")[["a", "pred"]].sum()
    return {"MAE mensual (celdas-día)": (p.pred - p.a).abs().mean(),
            "error anual (%)": ((yearly.pred - yearly.a).abs() / yearly.a.clip(lower=1)).mean() * 100,
            "r": np.corrcoef(p.pred, p.a)[0, 1]}


loyo = pd.DataFrame({(res, kind): evaluate(c, kind, CALIB_YEARS, CALIB_YEARS, leave_one_out=True)
                     for res, c in counts.items() for kind in KINDS}).T
loyo.index.names = ["grilla", "factor"]
loyo.round(3)

### Vista previa del holdout 2022–2025

Calibra en 2012–2021 y predice Aqua en 2022–2025. Si el error crece mucho respecto de la tabla anterior, es la deriva orbital de Aqua: va a la validación y al pitch, no es un error del método.

In [ ]:
present = {c.year.max() for c in counts.values()}
if max(present) < min(HOLDOUT_YEARS):
    print("Faltan datos de 2022–2025 para el holdout.")
else:
    holdout = pd.DataFrame({(res, kind): evaluate(c, kind, CALIB_YEARS, HOLDOUT_YEARS, leave_one_out=False)
                            for res, c in counts.items() for kind in KINDS}).T
    holdout.index.names = ["grilla", "factor"]
    display(holdout.round(3))

## Decisión

Completar **solo** con las tablas de arriba, después de correr el cuaderno con datos reales. Copiar la decisión al pull request y avisar a Datos (4) para `harmonize.py`.

| Decisión | Valor | Motivo (cifra de la tabla) |
|---|---|---|
| Ventana de calibración | pendiente | pendiente |
| Grilla para el factor | pendiente | pendiente |
| Tipo de factor | pendiente | pendiente |
| k medio de Argentina | pendiente | pendiente |

**Pendiente para después:** factor por región (provincia) con contracción hacia el nacional, cuando estén los polígonos de `ml/data/boundaries/` (Base de datos, 5).